# Read and Clean Dataset


In [1]:
# Set excel file to read
read_file = 'site_leaching_data_upd_b.xlsx'

# Set csv file to write
write_file = 'site_leaching_data.csv'

## 📚 Step 0: Import libraries

Load core libraries for data handling, modelling, and visualisation.

In [2]:
import pandas as pd
import numpy as np
import re

## 📥 Step 1: Load and clean column names

Import the dataset and standardise column names for consistency.

In [3]:
# Load dataset
df = pd.read_excel(read_file, sheet_name=0)

# Clean column names: strip whitespace and unify format
def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'\s+', '_', col)
    col = re.sub(r'[^\w\d_]', '', col)
    # Remove double underscores
    col = re.sub(r'__+', '_', col)
    # Convert to lowercase
    col = col.lower()
    # Capitablize first letter and each letter after an underscore
    col = col.replace('_', ' ').title().replace(' ', '_')
    return col

df.columns = [clean_column_name(col) for col in df.columns]

## ⚙️ Step 2: Define constants

Set plant configuration, material properties, modelling parameters, and economic assumptions.

In [4]:
# Plant configuration constants
PRELEACH_TANKS = 2           	# number of pre-leaching tanks
CIL_TANKS = 9                 	# number of CIL tanks
PRELEACH_TANK_VOLUE = 1000.0  	# volume of the pre-leaching tanks in m^3
CIL_TANK_VOLUME = 250.0        	# volume of the CIL tanks in m^3

# Material characteristics
SOLIDS_DENSITY = 2.70
SOLUTION_DENSITY = 1.10

# Cyanide leaching parameters
min_cn_hist = 100 			    # min. CN in ppm for cleansing historicals
max_cn_hist = 1200 				# max. CN in ppm for cleansing historicals
do_smoothing_window = 45		# smoothing window in days for DO data

# Feed charateristics
ultra_fine_sizing = 50	 		# midpoint for < 75 µm sizing
fine_sizing = 112.5				# midpoint for > 75 µm sizing
coarse_sizing = 175 			# midpoint for > 150 µm sizing

## 🧹 Step 3: Clean and engineer features

Prepare key fields for modelling by cleansing inputs, imputing missing values, and calculating derived features like solution mass, CN dosage, DO smoothing, particle size, and recovery.

In [5]:
# Define a function to calculate the solution mass
def calc_solution_mass(percent_solids, tonnes_processed):
    """
    Calculate the mass of the solution in kg based on the percentage of solids and tonnes processed.
    :param percent_solids: Percentage of solids in the slurry.
    :param tonnes_processed: Tonnes of material processed.
    :return: Mass of the solution in kg.
    """
    solids_fraction = percent_solids / 100
    slurry_mass_kg = tonnes_processed * 1000 / solids_fraction
    return slurry_mass_kg * (1 - solids_fraction)

# --- Calculate residence time across leach circuit ---
df["Residence_Time"] = (PRELEACH_TANKS * PRELEACH_TANK_VOLUE + 
                           CIL_TANKS * CIL_TANK_VOLUME) / df["Leach_Feed_Throughput_M3Hr"]

# --- Calculate daily throughput in tonnes ---
# Convert m³/day to tonnes/day using density and solids fraction
df["Tonnes_Processed"] = (
    df["Leach_Feed_Throughput_M3Day"] * (df["Percentsolids"] / 100) * SOLIDS_DENSITY
)

# --- Calculate CN dosage into leaching circuit ---	
# Fill NaNs with 0 for safe comparison
ppm_tank_1 = df["Cyanide_Profile_Ppm_Leach_Tank_1"].fillna(0).where(
lambda x: (x >= min_cn_hist) & (x <= max_cn_hist), 0
)
ppm_tank_2 = df["Cyanide_Profile_Ppm_Leach_Tank_2"].fillna(0).where(
    lambda x: (x >= min_cn_hist) & (x <= max_cn_hist), 0
)

# Flags indicating which pre-leach tank is receiving CN
df["Leach_Cn_Adds_To_Tank_1"] = ppm_tank_1 > ppm_tank_2
df["Leach_Cn_Adds_To_Tank_2"] = ~df["Leach_Cn_Adds_To_Tank_1"]

# Actual ppm value from the pre-leach tank identified as dosing
df["Leach_Cn_Adds_ppm"] = np.where(df["Leach_Cn_Adds_To_Tank_1"], ppm_tank_1, ppm_tank_2)


# % solids to fractional solids
df["Solids_Fraction"] = df["Percentsolids"] / 100
df["Liquid_Fraction"] = 1 - df["Solids_Fraction"]

# Wet mass of slurry (kg)
df["Slurry_Mass_Kg"] = df["Daily_Milledtreated_Tons"] * 1000 / df["Solids_Fraction"]

# Liquid mass in slurry (assuming water ~1kg/L)
df["Solution_Mass_Kg"] = df["Slurry_Mass_Kg"] * df["Liquid_Fraction"]

df["NaCN_Used_Kg"] = df["Leach_Cn_Adds_ppm"] * df["Solution_Mass_Kg"] / 1e6

df['CN_Kg_T'] = df['NaCN_Used_Kg'] / df['Daily_Milledtreated_Tons']

# --- Clean and impute dissolved oxygen (DO) profile ---
o2_col = "Dissolved_Oxygen_Profile_Ppm_Leach_Tank_01"

# Replace zero with NaN
df[o2_col] = df[o2_col].replace(0.0, np.nan)

# Impute using a rolling median (or fallback to overall median)
df[o2_col + "_Imputed"] = df[o2_col].fillna(
    df[o2_col].rolling(window=3, min_periods=1, center=True).median()
)

df["Dissolved_Oxygen_Profile_Ppm_Leach_Tank_01_Smooth"] = (
    df["Dissolved_Oxygen_Profile_Ppm_Leach_Tank_01_Imputed"]
    .rolling(window=do_smoothing_window, min_periods=1, center=True)
    .mean()
)

# Fallback for any remaining NaNs (start/end of window)
df[o2_col + "_Imputed"] = df[o2_col + "_Imputed"].fillna(df[o2_col].median())

# --- Standardise feed grade and throughput column names ---
df["Leach_Feed_Grade_Au_Gt"] = df["Leach_Feed_Grade_Au_Gt_Day"]

# Standardise particle size fields
df = df.rename(columns={
    "Leach_Feed_Gt_150Μm_Day": "Leach_Feed_Gt_150um_Day",
    "Leach_Feed_Gt_75Μm_Day": "Leach_Feed_Gt_75um_Day",
    "Leach_Feed_Lt_75Μm_Day": "Leach_Feed_Lt_75um_Day"
})

# Ensure datetime is parsed
df["Date"] = pd.to_datetime(df["Date"])


# --- Create Interaction terms ---
size_sum = (
        df["Leach_Feed_Lt_75um_Day"] +
        df["Leach_Feed_Gt_75um_Day"] +
        df["Leach_Feed_Gt_150um_Day"]
    )
df["Fines_Fraction"] = df["Leach_Feed_Lt_75um_Day"] / size_sum.replace(0, np.nan)
df["Grade_x_Fines"] = df["Leach_Feed_Grade_Au_Gt_Day"] * df["Fines_Fraction"]

ultra_fine_field = "Leach_Feed_Lt_75um_Day"
fine_field = "Leach_Feed_Gt_75um_Day"
coarse_field = "Leach_Feed_Gt_150um_Day"


# Calculate Effective Particle Size (um)
def get_effective_particle_size(row, ultra_field, fine_field, coarse_field,
                                ultra_size, fine_size, coarse_size):
    total = row[ultra_field] + row[fine_field] + row[coarse_field]
    if total == 0:
        return np.nan
    weighted_d = (
        row[ultra_field] * ultra_size +
        row[fine_field] * fine_size +
        row[coarse_field] * coarse_size
    ) / total
    return weighted_d

df["Effective_Particle_Size_um"] = df.apply(
    lambda row: get_effective_particle_size(
        row,
        ultra_fine_field,
        fine_field,
        coarse_field,
        ultra_fine_sizing,
        fine_sizing,
        coarse_sizing
    ),
    axis=1
)

# Calculate Au Recovery
df["Au_Recovery_g"] = (
    df["Cil_Feed_Total_Au_Grams"] - df["Total_Au_Tailing_Grams"]
)
df["Au_Recovery_pct"] = (df["Au_Recovery_g"] / df["Cil_Feed_Total_Au_Grams"]) * 100

df['Total_Au_in_Solution_g'] = (
    df['Dissolved_Au_Metal_Profile_G_Leach_Tank_01'] +
    df['Dissolved_Au_Metal_Profile_G_Leach_Tank_02'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_01'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_02'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_03'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_04'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_05'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_06'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_07'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_08'] +
    df['Dissolved_Au_Metal_Profile_G_Cil_Tank_09']
)

# Calculate Au Residuals
df['Au_Residual_g'] = df['Total_Au_in_Solution_g'] - (
    df['Au_Recovery_g'] + df['Total_Au_Tailing_Grams']
)

# Calculate Au Accounting
df['Au_Accounted_g'] = (
    df['Gold_On_Carbon_Grams'] + df['Total_Au_in_Solution_g'] +
    df['Total_Au_Tailing_Grams'] + df['Total_Undissolved_Au_Grams']
)

df["Au_Unaccounted_g"] = df["Cil_Feed_Total_Au_Grams"] - df["Au_Accounted_g"]

# Calculate Au Inventory in System
df['Au_Inventory_g'] = (
    df['Total_Au_in_Solution_g'] + df['Au_Residual_g'] + df['Total_Au_Tailing_Grams']
)

# --- Rename essential columns for clarity ---
df = df.rename(columns={
    "Cil_Feed_Total_Au_Grams": "Au_Feed_g",
    "Total_Au_Tailing_Grams": "Au_Tailings_g",
    "Cn_Conc_Tailing_Ppm": "CN_Tailings_Concentration",
    "Dissolved_Oxygen_Profile_Ppm_Leach_Tank_01_Smooth": "Dissolved_Oxygen_PL_01",
    "Effective_Particle_Size_um": "Particle_Size",
    "Leach_Cn_Adds_ppm": "CN_Concentration_PL_01",
    "Leach_Feed_Grade_Au_Gt_Day": "Au_Grade",
    "Daily_Milledtreated_Tons": "Leach_Feed_Dry_t",
    "Gold_On_Carbon_Grams": "Au_on_Carbon_g",
    "Total_Undissolved_Au_Grams": "Undissolved_Au_g",
})

In [6]:
# Save dataframe to csv
df.to_csv(write_file, index=False)